# NSL-KDD Pipeline
Self-contained end-to-end pipeline for training the MCL-FWA-BILSTM hybrid model on the NSL-KDD dataset.

**Stages:**
1. Architecture definitions (MCL, CustomBiLSTM, FWA, Extractor)
2. Training utilities (TrainConfig, train_extractor)
3. Classifier utilities (TreeConfig, fit_rf, fit_xgb, evaluate)
4. Data preprocessing (NSL-KDD → 120 features → 10×12 spatial tensor)
5. Phase 1: Train DL feature extractor
6. Phase 2: Train Random Forest & XGBoost on extracted features
7. Evaluate and compare results

## 1. Imports

In [ ]:
!nvidia-smi

In [ ]:
from __future__ import annotations

import os, sys
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from xgboost import XGBClassifier

print(f"PyTorch: {torch.__version__}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")

## 2. Architecture — Layer: MCL (CNN with Prediction-Error-Filter Constraint)

Paper §F, Algorithm 1, Eqs 1–4.  
Input: `(N, 10, 12)` spatial tensor. Output: `Wc` feature vector `(N, 56)`.

In [ ]:
class MCL(nn.Module):
    def __init__(self, in_features: int = 120, n_filters: int = 10, out_features: int = 56) -> None:
        super().__init__()
        # Input is (N, 1, 10, 12)
        self.mcl_conv = nn.Conv2d(1, n_filters, kernel_size=3, padding=1, bias=False)
        self.conv_stack = nn.Sequential(
            nn.Conv2d(n_filters, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # -> (N, 32, 5, 6)
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.flatten_dim = 16 * 5 * 6
        self.project = nn.Linear(self.flatten_dim, out_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N, 10, 12) -> (N, 1, 10, 12)
        h = x.unsqueeze(1)
        h = torch.tanh(self.mcl_conv(h))   # (N, 10, 10, 12)
        h = self.conv_stack(h)             # (N, 16, 5, 6)
        h = h.flatten(1)                   # (N, 480)
        return self.project(h)             # (N, 56)

    @torch.no_grad()
    def apply_mcl_constraint(self) -> None:
        """Re-project MCL conv weights to satisfy the zero-sum prediction-error-filter constraint."""
        w = self.mcl_conv.weight.data                      # (10, 1, 3, 3)
        w_flat = w.view(w.shape[0], w.shape[1], -1)        # (10, 1, 9)
        k = w_flat.shape[-1]
        center = k // 2
        idx = [i for i in range(k) if i != center]
        idx_t = torch.tensor(idx, device=w.device)
        others = w_flat.index_select(-1, idx_t)
        denom = others.abs().sum(dim=-1, keepdim=True).clamp_min(1e-8)
        normalized = others / denom
        w_flat.index_copy_(-1, idx_t, normalized)
        w_flat[..., center] = -normalized.sum(dim=-1)
        w.copy_(w_flat.view(w.shape))

## 3. Architecture — Layer: CustomBiLSTM (Fused CNN+RNN Cell)

Custom Bidirectional RNN that injects the MCL `Wc` feature into each timestep's update gate.

In [ ]:
class CustomRNNCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, wc_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_x = nn.Linear(input_size, hidden_size, bias=False)
        self.U_c = nn.Linear(hidden_size, hidden_size, bias=True)
        self.W_c_proj = nn.Linear(wc_size, hidden_size, bias=False)

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor, w_c: torch.Tensor) -> torch.Tensor:
        # F_B = tanh(W_c * F_t + U_c * F_t-1 + b_c)  (paper mapping)
        return torch.tanh(self.W_x(x_t) + self.U_c(h_prev) + self.W_c_proj(w_c))


class CustomBiLSTM(nn.Module):
    def __init__(self, input_size: int = 12, hidden_size: int = 64, wc_size: int = 56):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell_fwd = CustomRNNCell(input_size, hidden_size, wc_size)
        self.cell_bwd = CustomRNNCell(input_size, hidden_size, wc_size)

    def forward(self, x: torch.Tensor, w_c: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # x: (N, T, input_size) i.e. (N, 10, 12)
        N, T, _ = x.size()
        device = x.device
        h_fwd = torch.zeros(N, self.hidden_size, device=device)
        h_bwd = torch.zeros(N, self.hidden_size, device=device)
        out_fwd, out_bwd = [], []
        for t in range(T):
            h_fwd = self.cell_fwd(x[:, t, :], h_fwd, w_c)
            out_fwd.append(h_fwd)
        for t in reversed(range(T)):
            h_bwd = self.cell_bwd(x[:, t, :], h_bwd, w_c)
            out_bwd.insert(0, h_bwd)
        out_fwd = torch.stack(out_fwd, dim=1)                           # (N, T, H)
        out_bwd = torch.stack(out_bwd, dim=1)                           # (N, T, H)
        out = torch.cat([out_fwd, out_bwd], dim=-1)                     # (N, T, 2H)
        fbl = torch.cat([out_fwd[:, -1, :], out_bwd[:, 0, :]], dim=-1) # (N, 2H)
        return out, fbl

## 4. Architecture — Layer: FWA (Feature-Weighted Attention)

Paper Algorithm 2, lines 395–405; Eqs 1–2.  
Bahdanau-style additive attention where `Wc` acts as the query over BiLSTM hidden states.

In [ ]:
class FWA(nn.Module):
    """
    score_t = v^T · tanh(W1·h_t + W2·Wc + b)
    FA_t    = softmax_t(score_t)
    FAT     = Σ_t FA_t · h_t
    Fe      = concat(FBL, FAT)
    """
    def __init__(self, hidden_dim: int, wc_dim: int, attn_dim: int | None = None):
        super().__init__()
        if attn_dim is None:
            attn_dim = hidden_dim
        self.W1 = nn.Linear(hidden_dim, attn_dim, bias=False)
        self.W2 = nn.Linear(wc_dim, attn_dim, bias=True)
        self.v  = nn.Linear(attn_dim, 1, bias=False)

    def forward(
        self,
        h:   torch.Tensor,   # (N, T, 2H)
        wc:  torch.Tensor,   # (N, wc_dim)
        fbl: torch.Tensor,   # (N, 2H)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        q      = self.W2(wc).unsqueeze(1)               # (N, 1, attn_dim)
        k      = self.W1(h)                             # (N, T, attn_dim)
        scores = self.v(torch.tanh(k + q)).squeeze(-1)  # (N, T)
        fa     = torch.softmax(scores, dim=-1)          # (N, T)
        fat    = torch.bmm(fa.unsqueeze(1), h).squeeze(1)  # (N, 2H)
        fe     = torch.cat([fbl, fat], dim=-1)          # (N, 4H)
        return fe, fa

## 5. Architecture — Extractor & ExtractorWithHead

`Extractor` chains MCL → CustomBiLSTM → FWA.  
`ExtractorWithHead` wraps it with a temporary linear classifier for Phase-1 training.

In [ ]:
class Extractor(nn.Module):
    def __init__(
        self,
        in_features:  int = 120,
        mcl_filters:  int = 10,
        wc_dim:       int = 56,
        lstm_hidden:  int = 64,
        attn_dim: int | None = None,
    ):
        super().__init__()
        self.in_features = in_features
        self.mcl    = MCL(in_features=in_features, n_filters=mcl_filters, out_features=wc_dim)
        self.bilstm = CustomBiLSTM(input_size=12, hidden_size=lstm_hidden, wc_size=wc_dim)
        self.fwa    = FWA(hidden_dim=2 * lstm_hidden, wc_dim=wc_dim, attn_dim=attn_dim)
        self.feature_dim = 4 * lstm_hidden

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, dict]:
        wc       = self.mcl(x)             # (N, wc_dim)
        h, fbl   = self.bilstm(x, wc)      # (N, T, 2H), (N, 2H)
        fe, fa   = self.fwa(h, wc, fbl)    # (N, 4H), (N, T)
        return fe, {"wc": wc, "fbl": fbl, "fa": fa}

    def apply_mcl_constraint(self) -> None:
        self.mcl.apply_mcl_constraint()


class ExtractorWithHead(nn.Module):
    """Phase-1 wrapper: extractor + temporary linear classifier head."""
    def __init__(self, extractor: Extractor, n_classes: int):
        super().__init__()
        self.extractor = extractor
        self.head = nn.Linear(extractor.feature_dim, n_classes)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        fe, _ = self.extractor(x)
        return self.head(fe), fe

    def apply_mcl_constraint(self) -> None:
        self.extractor.apply_mcl_constraint()

## 6. Training Utilities

In [ ]:
@dataclass
class TrainConfig:
    epochs:                  int   = 100
    batch_size:              int   = 256
    lr:                      float = 1e-3
    weight_decay:            float = 0.0
    device:                  str   = "cuda" if torch.cuda.is_available() else "cpu"
    target:                  str   = "binary"   # "binary" | "multi"
    log_every:               int   = 0          # 0 = silent
    seed:                    int   = 0
    extractor_kwargs:        dict  = field(default_factory=dict)
    target_val_acc:          float = 0.90
    early_stopping_patience: int   = 5


def _make_loader(X: torch.Tensor, y: torch.Tensor, cfg: TrainConfig, shuffle: bool) -> DataLoader:
    ds = TensorDataset(X, y.long())
    return DataLoader(ds, batch_size=cfg.batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=cfg.device.startswith("cuda"))


@torch.no_grad()
def _evaluate_loop(model, loader, loss_fn, device) -> dict:
    model.eval()
    running, seen, correct = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        logits, _ = model(xb)
        loss = loss_fn(logits, yb)
        bs = yb.size(0)
        running += loss.item() * bs
        seen    += bs
        correct += (logits.argmax(dim=-1) == yb).sum().item()
    return {"val_loss": running / seen, "val_acc": correct / seen}


def train_extractor(
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    cfg: TrainConfig | None = None,
    X_val: torch.Tensor | None = None,
    y_val: torch.Tensor | None = None,
) -> tuple[Extractor, list[dict]]:
    cfg = cfg or TrainConfig()
    torch.manual_seed(cfg.seed)
    n_classes = int(y_train.max().item()) + 1
    extractor = Extractor(in_features=X_train.shape[1], **cfg.extractor_kwargs)
    model     = ExtractorWithHead(extractor, n_classes=n_classes).to(cfg.device)
    if cfg.device.startswith("cuda") and torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)
    opt     = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    train_loader = _make_loader(X_train, y_train, cfg, shuffle=True)
    val_loader   = _make_loader(X_val, y_val, cfg, shuffle=False) if X_val is not None else None
    history, best_val_loss, patience_counter = [], float("inf"), 0

    for epoch in range(cfg.epochs):
        model.train()
        running, seen, correct = 0.0, 0, 0
        for step, (xb, yb) in enumerate(train_loader):
            xb, yb = xb.to(cfg.device, non_blocking=True), yb.to(cfg.device, non_blocking=True)
            logits, _ = model(xb)
            loss = loss_fn(logits, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            underlying = model.module if isinstance(model, torch.nn.DataParallel) else model
            underlying.apply_mcl_constraint()
            bs = yb.size(0)
            running += loss.item() * bs; seen += bs
            correct += (logits.argmax(dim=-1) == yb).sum().item()
            if cfg.log_every and (step + 1) % cfg.log_every == 0:
                print(f"epoch {epoch} step {step+1} loss {running/seen:.4f}")
        row = {"epoch": epoch, "train_loss": running / seen, "train_acc": correct / seen}
        if val_loader is not None:
            val_metrics = _evaluate_loop(model, val_loader, loss_fn, cfg.device)
            row.update(val_metrics)
        history.append(row)
        if cfg.log_every: print(row)
        if val_loader is not None:
            val_acc, val_loss = val_metrics["val_acc"], val_metrics["val_loss"]
            if val_acc >= cfg.target_val_acc:
                print(f"Reached target validation accuracy: {val_acc:.4f} >= {cfg.target_val_acc}. Stopping.")
                break
            if val_loss < best_val_loss:
                best_val_loss = val_loss; patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= cfg.early_stopping_patience:
                    print(f"Validation loss did not improve for {cfg.early_stopping_patience} epochs.")
    return extractor, history


@torch.no_grad()
def extract_features(
    extractor: Extractor, X: torch.Tensor, batch_size: int = 512, device: str | None = None
) -> torch.Tensor:
    device = device or next(extractor.parameters()).device.type
    extractor.eval()
    out = []
    for i in range(0, X.shape[0], batch_size):
        xb = X[i : i + batch_size].to(device, non_blocking=True)
        fe, _ = extractor(xb)
        out.append(fe.cpu())
    return torch.cat(out, dim=0)

## 7. Classifier Utilities (Random Forest & XGBoost)

In [ ]:
@dataclass
class TreeConfig:
    n_estimators: int       = 200
    max_depth:    int | None = None
    n_jobs:       int       = -1
    random_state: int       = 0
    extra:        dict      = field(default_factory=dict)


def fit_rf(Fe_train: np.ndarray, y_train: np.ndarray, cfg: TreeConfig | None = None) -> RandomForestClassifier:
    cfg = cfg or TreeConfig()
    clf = RandomForestClassifier(
        n_estimators=cfg.n_estimators, max_depth=cfg.max_depth,
        n_jobs=cfg.n_jobs, class_weight="balanced",
        random_state=cfg.random_state, **cfg.extra,
    )
    clf.fit(Fe_train, y_train)
    return clf


def fit_xgb(Fe_train: np.ndarray, y_train: np.ndarray, cfg: TreeConfig | None = None) -> XGBClassifier:
    cfg = cfg or TreeConfig()
    clf = XGBClassifier(
        n_estimators=cfg.n_estimators, max_depth=cfg.max_depth,
        n_jobs=cfg.n_jobs, random_state=cfg.random_state,
        eval_metric="mlogloss", **cfg.extra,
    )
    clf.fit(Fe_train, y_train)
    return clf


def evaluate(clf, Fe: np.ndarray, y: np.ndarray) -> dict:
    y_pred = clf.predict(Fe)
    avg = "binary" if len(np.unique(y)) == 2 else "macro"
    return {
        "accuracy":         accuracy_score(y, y_pred),
        "precision":        precision_score(y, y_pred, average=avg, zero_division=0),
        "recall":           recall_score(y, y_pred, average=avg, zero_division=0),
        "f1":               f1_score(y, y_pred, average=avg, zero_division=0),
        "confusion_matrix": confusion_matrix(y, y_pred).tolist(),
        "report":           classification_report(y, y_pred, zero_division=0),
    }

## 8. Data Preprocessing — NSL-KDD

- Encodes 3 categorical columns (`protocol_type`, `service`, `flag`) via one-hot → 120 features
- Applies `StandardScaler`
- Reshapes to `(N, 10, 12)` for the CNN-MCL spatial input

In [ ]:
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty",
]
CATEGORICAL   = ["protocol_type", "service", "flag"]
DROP_CONSTANT = ["num_outbound_cmds", "is_host_login"]

ATTACK_FAMILY = {
    "normal": "Normal",
    "back": "DoS", "land": "DoS", "neptune": "DoS", "pod": "DoS", "smurf": "DoS",
    "teardrop": "DoS", "apache2": "DoS", "udpstorm": "DoS", "processtable": "DoS",
    "worm": "DoS", "mailbomb": "DoS",
    "ipsweep": "Probe", "nmap": "Probe", "portsweep": "Probe", "satan": "Probe",
    "mscan": "Probe", "saint": "Probe",
    "ftp_write": "R2L", "guess_passwd": "R2L", "imap": "R2L", "multihop": "R2L",
    "phf": "R2L", "spy": "R2L", "warezclient": "R2L", "warezmaster": "R2L",
    "sendmail": "R2L", "named": "R2L", "snmpgetattack": "R2L", "snmpguess": "R2L",
    "xlock": "R2L", "xsnoop": "R2L", "httptunnel": "R2L",
    "buffer_overflow": "U2R", "loadmodule": "U2R", "perl": "U2R", "rootkit": "U2R",
    "ps": "U2R", "sqlattack": "U2R", "xterm": "U2R",
}
FAMILY_IDX = {"Normal": 0, "DoS": 1, "Probe": 2, "R2L": 3, "U2R": 4}


def load_raw_nsl(path: str | Path) -> pd.DataFrame:
    return pd.read_csv(path, header=None, names=COLUMNS)


def preprocess_nsl(train_df: pd.DataFrame, test_df: pd.DataFrame):
    train_df = train_df.drop(columns=["difficulty"]).copy()
    test_df  = test_df.drop(columns=["difficulty"]).copy()
    for col in DROP_CONSTANT:
        train_df = train_df.drop(columns=col)
        test_df  = test_df.drop(columns=col)
    y_bin_train = (train_df["label"] != "normal").astype(np.int64).to_numpy()
    y_bin_test  = (test_df["label"]  != "normal").astype(np.int64).to_numpy()
    fam = lambda lbl: FAMILY_IDX[ATTACK_FAMILY.get(lbl, "R2L")]
    y_mul_train = train_df["label"].map(fam).to_numpy(dtype=np.int64)
    y_mul_test  = test_df["label"].map(fam).to_numpy(dtype=np.int64)
    train_df = train_df.drop(columns="label")
    test_df  = test_df.drop(columns="label")
    combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    combined = pd.get_dummies(combined, columns=CATEGORICAL, dtype=np.float32)
    feature_names = combined.columns.tolist()
    n_train  = len(train_df)
    X_train  = combined.iloc[:n_train].to_numpy(dtype=np.float32)
    X_test   = combined.iloc[n_train:].to_numpy(dtype=np.float32)
    scaler   = StandardScaler()
    X_train  = scaler.fit_transform(X_train).astype(np.float32)
    X_test   = scaler.transform(X_test).astype(np.float32)
    if X_train.shape[1] == 120:
        X_train = X_train.reshape(-1, 10, 12)
        X_test  = X_test.reshape(-1, 10, 12)
    else:
        print(f"Warning: Expected 120 features, got {X_train.shape[1]}. Reshaping skipped.")
    return X_train, X_test, y_bin_train, y_bin_test, y_mul_train, y_mul_test, feature_names


def build_and_save_nsl(train_path, test_path, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    train_df = load_raw_nsl(train_path)
    test_df  = load_raw_nsl(test_path)
    X_tr, X_te, yb_tr, yb_te, ym_tr, ym_te, feats = preprocess_nsl(train_df, test_df)
    torch.save({"X": torch.from_numpy(X_tr), "y_bin": torch.from_numpy(yb_tr),
                "y_mul": torch.from_numpy(ym_tr), "feature_names": feats}, out_dir / "train.pt")
    torch.save({"X": torch.from_numpy(X_te), "y_bin": torch.from_numpy(yb_te),
                "y_mul": torch.from_numpy(ym_te), "feature_names": feats}, out_dir / "test.pt")
    print(f"Train shape: {X_tr.shape}, Test shape: {X_te.shape}")

def load_processed(path) -> dict:
    return torch.load(path, weights_only=False)

## 9. Paths Configuration

In [ ]:
BASE_DIR   = Path(".").resolve().parent
## BASE_DIR = "/mnt/c/Users/sduai/Documents/projis/research/nids_dl/nids-research"
TRAIN_PATH = BASE_DIR / "NSL-KDD" / "KDDTrain+.txt"
TEST_PATH  = BASE_DIR / "NSL-KDD" / "KDDTest+.txt"
OUT_DIR    = BASE_DIR / "data" / "processed"

print(f"Base dir:   {BASE_DIR}")
print(f"Train path: {TRAIN_PATH}")
print(f"Test path:  {TEST_PATH}")

## 10. Preprocess & Load Data

In [ ]:
train_pt = OUT_DIR / "train.pt"
if not train_pt.exists():
    print("Preprocessing data...")
    build_and_save_nsl(TRAIN_PATH, TEST_PATH, OUT_DIR)
else:
    print("Cached data found. Skipping preprocessing.")

train_data = load_processed(OUT_DIR / "train.pt")
test_data  = load_processed(OUT_DIR / "test.pt")

X_tr, y_bin_tr = train_data["X"], train_data["y_bin"]
X_te, y_bin_te = test_data["X"],  test_data["y_bin"]

print(f"Train X shape : {X_tr.shape}")
print(f"Test  X shape : {X_te.shape}")
print(f"Train: Benign={int((y_bin_tr==0).sum())}, Attack={int((y_bin_tr==1).sum())}")
print(f"Test : Benign={int((y_bin_te==0).sum())}, Attack={int((y_bin_te==1).sum())}")

## 11. Phase 1 — Train DL Feature Extractor

In [ ]:
EPOCH_N = 100

cfg = TrainConfig(
    epochs=EPOCH_N, batch_size=256, device=DEVICE,
    log_every=10, target_val_acc=0.90, early_stopping_patience=20,
)

print(f"Training for up to {EPOCH_N} epochs on {DEVICE}...")
extractor, hist = train_extractor(X_tr, y_bin_tr, cfg, X_val=X_te, y_val=y_bin_te)

## 12. Feature Extraction

In [ ]:
fe_tr = extract_features(extractor, X_tr, device=cfg.device).numpy()
fe_te = extract_features(extractor, X_te, device=cfg.device).numpy()
print(f"Train features: {fe_tr.shape}, Test features: {fe_te.shape}")

## 13. Phase 2 — Train & Evaluate Classifiers

In [ ]:
print("Training Random Forest...")
rf_clf     = fit_rf(fe_tr, y_bin_tr.numpy(), TreeConfig(n_estimators=100))
rf_metrics = evaluate(rf_clf, fe_te, y_bin_te.numpy())

print("Training XGBoost...")
xgb_clf     = fit_xgb(fe_tr, y_bin_tr.numpy(), TreeConfig(n_estimators=100))
xgb_metrics = evaluate(xgb_clf, fe_te, y_bin_te.numpy())

## 14. Results

In [ ]:
results = pd.DataFrame([
    {"Classifier": "Random Forest", **{k: v for k, v in rf_metrics.items()  if k not in ("confusion_matrix", "report")}},
    {"Classifier": "XGBoost",       **{k: v for k, v in xgb_metrics.items() if k not in ("confusion_matrix", "report")}},
]).set_index("Classifier").round(4)
results.columns = [c.capitalize() for c in results.columns]

print("\n=== NSL-KDD Results ===")
print(results.to_string())
results

In [ ]:
print("\n--- Random Forest Classification Report ---")
print(rf_metrics["report"])
print("\n--- XGBoost Classification Report ---")
print(xgb_metrics["report"])

## 15. Visualizations

All plots for the paper: training curves, FWA attention, classifier comparison, confusion matrices, ROC, PR, t-SNE, RF feature importance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.manifold import TSNE

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
COLORS = {'RF': '#4C72B0', 'XGB': '#DD8452'}
print('Visualization imports ready.')

### V1 — DL Training Curves (Loss & Accuracy)

In [ ]:
hist_df = pd.DataFrame(hist)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', linewidth=2)
if 'val_loss' in hist_df:
    ax1.plot(hist_df['epoch'], hist_df['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
ax1.set_title('Training Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss'); ax1.legend()

ax2.plot(hist_df['epoch'], hist_df['train_acc'], label='Train Acc', linewidth=2)
if 'val_acc' in hist_df:
    ax2.plot(hist_df['epoch'], hist_df['val_acc'], label='Val Acc', linewidth=2, linestyle='--')
ax2.axhline(0.90, color='red', linestyle=':', alpha=0.7, label='Target 90%')
ax2.set_title('Validation Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()

plt.suptitle('MCL-FWA-BiLSTM Training', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.pdf', bbox_inches='tight', dpi=300)
plt.show()

### V2 — FWA Attention Weights (Spatial Row Significance)

In [ ]:
@torch.no_grad()
def get_attention_weights(extractor, X, device, batch_size=512):
    extractor.eval()
    all_fa = []
    for i in range(0, X.shape[0], batch_size):
        xb = X[i:i+batch_size].to(device, non_blocking=True)
        _, meta = extractor(xb)
        all_fa.append(meta['fa'].cpu())
    return torch.cat(all_fa, dim=0).numpy()  # (N, T=10)

fa_te     = get_attention_weights(extractor, X_te, cfg.device)
fa_benign = fa_te[y_bin_te.numpy() == 0].mean(axis=0)
fa_attack = fa_te[y_bin_te.numpy() == 1].mean(axis=0)

x_pos = np.arange(10); width = 0.35
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(x_pos - width/2, fa_benign, width, label='Benign',  color='#4C72B0', alpha=0.85)
ax.bar(x_pos + width/2, fa_attack,  width, label='Attack', color='#DD8452', alpha=0.85)
ax.set_xticks(x_pos); ax.set_xticklabels([f'Row {i+1}' for i in range(10)])
ax.set_xlabel('Spatial Row (10x12 feature map)')
ax.set_ylabel('Mean FWA Attention Weight')
ax.set_title('FWA Attention Weights by Class', fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('fwa_attention.pdf', bbox_inches='tight', dpi=300)
plt.show()

### V3 — Classifier Comparison Bar Chart (Acc / Precision / Recall / F1)

In [ ]:
metrics_keys = ['accuracy', 'precision', 'recall', 'f1']
rf_vals  = [rf_metrics[k]  for k in metrics_keys]
xgb_vals = [xgb_metrics[k] for k in metrics_keys]
x = np.arange(len(metrics_keys)); width = 0.35

fig, ax = plt.subplots(figsize=(9, 4.5))
bars_rf  = ax.bar(x - width/2, rf_vals,  width, label='Random Forest', color=COLORS['RF'])
bars_xgb = ax.bar(x + width/2, xgb_vals, width, label='XGBoost',       color=COLORS['XGB'])
for bar in list(bars_rf) + list(bars_xgb):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.002, f'{h:.4f}',
            ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([m.capitalize() for m in metrics_keys])
ax.set_ylim(0.85, 1.02); ax.set_ylabel('Score')
ax.set_title('RF vs XGBoost — Classifier Comparison', fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('classifier_comparison.pdf', bbox_inches='tight', dpi=300)
plt.show()

### V4 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
class_labels = ['Benign', 'Attack']

for ax, (name, metrics) in zip(axes, [
    ('Random Forest', rf_metrics),
    ('XGBoost',       xgb_metrics),
]):
    cm = np.array(metrics['confusion_matrix'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_pct, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels,
                ax=ax, cbar=False, linewidths=0.5)
    ax.set_title(f'Confusion Matrix — {name}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('confusion_matrices.pdf', bbox_inches='tight', dpi=300)
plt.show()

### V5 — ROC Curves with AUC

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
y_true = y_bin_te.numpy()

for name, clf, color in [
    ('Random Forest', rf_clf,  COLORS['RF']),
    ('XGBoost',       xgb_clf, COLORS['XGB']),
]:
    probs = clf.predict_proba(fe_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_true, probs)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random Baseline')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(loc='lower right'); plt.tight_layout()
plt.savefig('roc_curve.pdf', bbox_inches='tight', dpi=300)
plt.show()

### V6 — Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))

for name, clf, color in [
    ('Random Forest', rf_clf,  COLORS['RF']),
    ('XGBoost',       xgb_clf, COLORS['XGB']),
]:
    probs = clf.predict_proba(fe_te)[:, 1]
    prec, rec, _ = precision_recall_curve(y_true, probs)
    pr_auc = auc(rec, prec)
    ax.plot(rec, prec, color=color, lw=2, label=f'{name} (AUC = {pr_auc:.4f})')

baseline = y_true.mean()
ax.axhline(baseline, color='k', linestyle='--', lw=1, alpha=0.5, label=f'Baseline ({baseline:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('pr_curve.pdf', bbox_inches='tight', dpi=300)
plt.show()

### V7 — t-SNE of DL-Extracted Feature Space

In [ ]:
N_TSNE = min(5000, len(fe_te))
rng = np.random.default_rng(42)
idx = rng.choice(len(fe_te), N_TSNE, replace=False)
print(f'Running t-SNE on {N_TSNE} test samples...')
tsne = TSNE(n_components=2, perplexity=40, random_state=42, n_jobs=-1)
emb  = tsne.fit_transform(fe_te[idx])

labels_sub = y_bin_te.numpy()[idx]
fig, ax = plt.subplots(figsize=(8, 6))
for cls, name, color in [(0, 'Benign', '#4C72B0'), (1, 'Attack', '#DD8452')]:
    mask = labels_sub == cls
    ax.scatter(emb[mask, 0], emb[mask, 1], s=6, alpha=0.5, color=color, label=name)
ax.set_title('t-SNE of DL Features', fontweight='bold')
ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
ax.legend(markerscale=4); plt.tight_layout()
plt.savefig('tsne.pdf', bbox_inches='tight', dpi=300)
plt.show()